# 🧠 PsycheOS — GRPO Training Notebook
**Meta PyTorch OpenEnv Hackathon × Scaler School of Technology**

Run this notebook on Google Colab with a T4/A100 GPU using your HuggingFace compute credits.

**What this does:**
- Loads LLaMA-3-8B with Unsloth (4-bit quantized)
- Trains with GRPO on PsycheOS patient episodes
- Shows live reward curves
- Saves LoRA adapters to HuggingFace Hub

In [ ]:
# Step 1: Install dependencies
!pip install unsloth trl peft transformers datasets openenv gymnasium -q
!pip install faiss-cpu sentence-transformers langgraph langchain -q
!pip install wandb matplotlib -q
print('✅ Dependencies installed')

In [ ]:
# Step 2: Clone PsycheOS from HuggingFace Spaces
!git clone https://huggingface.co/spaces/mesazephyr/psycheos
%cd psycheos
import sys
sys.path.insert(0, '.')
print('✅ PsycheOS loaded')

In [ ]:
# Step 3: Load model with Unsloth
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/llama-3-8b-instruct',
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing=True,
)
print('✅ Model loaded with LoRA adapters')

In [ ]:
# Step 4: Generate training episodes
from environment.psycheos_env import PsycheOSEnv
from reward.reward_functions import composite_reward
import json

env = PsycheOSEnv()
episodes = env.generate_episodes(n=1000)
print(f'✅ Generated {len(episodes)} episodes')

# Show distribution
from collections import Counter
dist = Counter(ep['true_distress'] for ep in episodes)
print('Distress distribution:', dict(sorted(dist.items())))

In [ ]:
# Step 5: Define reward function for GRPO
def psycheos_reward_fn(response: str, episode: dict) -> float:
    try:
        clean = response.strip().replace('```json', '').replace('```', '')
        parsed = json.loads(clean)
        pred_level = int(parsed.get('distress_level', 1))
        resp_text = parsed.get('response', '')
        escalated = bool(parsed.get('escalate', False))
    except Exception:
        return -1.0

    rewards = composite_reward(
        pred_level=pred_level,
        true_level=episode['true_distress'],
        response=resp_text,
        escalated=escalated,
        ground_truth_escalate=episode['ground_truth_escalate'],
        distress_level=pred_level,
    )
    return rewards['composite']

print('✅ Reward function ready')

In [ ]:
# Step 6: Format dataset for GRPO
def format_prompt(episode):
    obs = episode['observation']
    return f"""You are a compassionate mental health support agent.

Patient message: {obs['message']}
Session: {obs['session']}

Respond ONLY with JSON:
{{\"distress_level\": <1-5>, \"response\": \"<empathetic response>\", \"escalate\": <true/false>}}"""

from datasets import Dataset

dataset = Dataset.from_list([
    {'prompt': format_prompt(ep), 'episode_json': json.dumps(ep)}
    for ep in episodes
])
print(f'✅ Dataset ready: {len(dataset)} samples')

In [ ]:
# Step 7: Run GRPO training
from trl import GRPOTrainer, GRPOConfig

config = GRPOConfig(
    output_dir='./psycheos_checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=200,
    warmup_ratio=0.1,
    bf16=True,
    report_to='wandb',
    run_name='psycheos-grpo',
)

trainer = GRPOTrainer(
    model=model,
    config=config,
    train_dataset=dataset,
    reward_funcs=[
        lambda outputs, refs: [
            psycheos_reward_fn(o, json.loads(r['episode_json']))
            for o, r in zip(outputs, refs)
        ]
    ],
)

print('🚀 Starting GRPO training...')
trainer.train()

In [ ]:
# Step 8: Evaluate and plot reward curves
from eval.reward_curves import evaluate_agent, plot_reward_curves

print('Evaluating...')
results = evaluate_agent(n_episodes=200)
print(json.dumps(results, indent=2))

plot_reward_curves(save_path='./reward_curves.png')

from IPython.display import Image
Image('./reward_curves.png')

In [ ]:
# Step 9: Push to HuggingFace Hub
from huggingface_hub import login
login()  # Enter your HF token

model.push_to_hub('mesazephyr/psycheos-lora')
tokenizer.push_to_hub('mesazephyr/psycheos-lora')
print('✅ Model pushed to HuggingFace Hub')